![](https://full-stack-assets.s3.eu-west-3.amazonaws.com/apple-logo.png)

# Apple Retail Store - From CSV to Mongo

## Business context

Apple's leadership wants a unified analytics backend in MongoDB to answer:

1. Revenue by **category** and **region** (monthly trend)
2. **Top products** by revenue (last 90 days)
3. **Store** performance month-over-month (MoM)
4. **Warranty** coverage & average duration by category
5. **ASP vs list price** sanity check by category

Data sources (CSV):

* `products.csv` — product catalog: `product_id, name, category_id, brand, list_price`
* `category.csv` — categories: `category_id, category_name, parent_id?`
* `stores.csv` — stores: `store_id, store_name, city, region, country`
* `sales.csv` — **big** fact table: `sale_id, product_id, store_id, date, units_sold, total_amount` (≥ 1M rows)
* `warranty.csv` — warranty info: `product_id, duration_months`


## Part I - Insert small data

In [ ]:
#!pip install -q xgboost
#!pip install -q s3fs
#!pip install -U kaleido
#!pip install Boto3
#!pip install s3fs
# Load in our libraries
#pip install pymongo[srv]
#pip install dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
# import s3fs
#import boto3

# plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.figure_factory as ff
import plotly.colors as pc

# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "colab" # pour que colab ne bloque pas l'export svg

import warnings
warnings.filterwarnings('ignore')



# montage drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1️⃣ Imports corrects
from google.colab import userdata
from pymongo import MongoClient
from pymongo.server_api import ServerApi

# 2️⃣ Installer pymongo DANS Colab (avant l'import si ce n'est pas déjà fait)
%pip install pymongo[srv]


In [ ]:

# 3️⃣ Récupération des secrets Colab
USERNAME = userdata.get("MONGO_USERNAME")
PASSWORD = userdata.get("PASSWORD")
CLUSTER = userdata.get("CLUSTER_NAME")


# Vérification explicite
if not all([USERNAME, PASSWORD, CLUSTER]):
    raise ValueError("Un ou plusieurs secrets Colab sont manquants")

print(f"Hello {CLUSTER}")

# URI MongoDB CORRECTE
MONGODB_URI = (
    f"mongodb+srv://{USERNAME}:{PASSWORD}@{CLUSTER}/"
    "?retryWrites=true&w=majority&appName=Cluster0"
)

# Connexion
client = MongoClient(MONGODB_URI, server_api=ServerApi("1"))

client.admin.command("ping")
print("Pinged your deployment. You successfully connected to MongoDB!")




Hello cluster0.u9zhept.mongodb.net
Pinged your deployment. You successfully connected to MongoDB!


1. Create a virtual environment and install dependencies.

2. Set your Mongo URI (Atlas or local):

For the following, we recommend organizing your project like so:

```
apple_exercise/
  data/
    products.csv
    category.csv
    stores.csv
    sales.csv
    warranty.csv
  apple_exercise.ipynb   # put the code from the next parts here
```

3. Using `pandas` load the following datasets:
    - `products`
    - `categories`
    - `stores`
    - `warranties`

In [ ]:
# Chargement des fichiers CSV (dimensions)
files = [

]
products = pd.read_csv("/content/drive/MyDrive/JedhaBootcamp/products.csv")
categories = pd.read_csv("/content/drive/MyDrive/JedhaBootcamp/categories.csv")
stores = pd.read_csv("/content/drive/MyDrive/JedhaBootcamp/stores.csv")
warranties = pd.read_csv("/content/drive/MyDrive/JedhaBootcamp/warranties.csv")

# Vérification rapide
print("Products shape:", products.shape)
print("Categories shape:", categories.shape)
print("Stores shape:", stores.shape)
print("Warranties shape:", warranties.shape)

# Aperçu des données
display(products.head())
display(categories.head())
display(stores.head())
display(warranties.head())


Products shape: (89, 5)
Categories shape: (10, 2)
Stores shape: (75, 4)
Warranties shape: (30000, 4)


,Product_ID,Product_Name,Category_ID,Launch_Date,Price
0,P-1,MacBook,CAT-1,2023-09-17,1149
1,P-2,MacBook Air (M1),CAT-1,2023-11-11,1783
2,P-3,MacBook Air (M2),CAT-1,2020-05-24,1588
3,P-4,MacBook Pro 13-inch,CAT-1,2021-01-17,1351
4,P-5,MacBook Pro 14-inch,CAT-1,2024-05-12,768


,category_id,category_name
0,CAT-1,Laptop
1,CAT-2,Audio
2,CAT-3,Tablet
3,CAT-4,Smartphone
4,CAT-5,Wearable


,Store_ID,Store_Name,City,Country
0,ST-1,Apple Fifth Avenue,New York,United States
1,ST-2,Apple Union Square,San Francisco,United States
2,ST-3,Apple Michigan Avenue,Chicago,United States
3,ST-4,Apple The Grove,Los Angeles,United States
4,ST-5,Apple SoHo,New York,United States


,claim_id,claim_date,sale_id,repair_status
0,CL-58750,2024-01-30,YG-8782,Completed
1,CL-8874,2024-06-25,QX-999001,Pending
2,CL-14486,2024-08-13,JG-46890,Pending
3,CL-42187,2024-09-19,XJ-1731,Pending
4,CL-37590,2024-09-16,FG-95080,Completed


4. Create helper function that:
    - Normalize columns data. Meaning:
        - all columns should be lower case
        - No spaces, only `_`
        - No special characters

    - Apply on all data

In [ ]:
import re

def normalize_columns(df):
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )
    return df


In [ ]:
products = normalize_columns(products)
categories = normalize_columns(categories)
stores = normalize_columns(stores)
warranties = normalize_columns(warranties)


In [ ]:
display(products.head())
display(categories.head())
display(stores.head())
display(warranties.head())

,product_id,product_name,category_id,launch_date,price
0,P-1,MacBook,CAT-1,2023-09-17,1149
1,P-2,MacBook Air (M1),CAT-1,2023-11-11,1783
2,P-3,MacBook Air (M2),CAT-1,2020-05-24,1588
3,P-4,MacBook Pro 13-inch,CAT-1,2021-01-17,1351
4,P-5,MacBook Pro 14-inch,CAT-1,2024-05-12,768


,category_id,category_name
0,CAT-1,Laptop
1,CAT-2,Audio
2,CAT-3,Tablet
3,CAT-4,Smartphone
4,CAT-5,Wearable


,store_id,store_name,city,country
0,ST-1,Apple Fifth Avenue,New York,United States
1,ST-2,Apple Union Square,San Francisco,United States
2,ST-3,Apple Michigan Avenue,Chicago,United States
3,ST-4,Apple The Grove,Los Angeles,United States
4,ST-5,Apple SoHo,New York,United States


,claim_id,claim_date,sale_id,repair_status
0,CL-58750,2024-01-30,YG-8782,Completed
1,CL-8874,2024-06-25,QX-999001,Pending
2,CL-14486,2024-08-13,JG-46890,Pending
3,CL-42187,2024-09-19,XJ-1731,Pending
4,CL-37590,2024-09-16,FG-95080,Completed


4. Cast columns of each columns to the most relevant datatype (i.e `str` or `float` etc.)

5. Create a new database called `apple_retail_store` with the right collections:

- `products`
- `categories`
- `stores`
- `warranties`

In [ ]:
client = MongoClient(MONGODB_URI)
db = client["apple_retail_store"]

from pymongo import MongoClient
MONGODB_URI

client = MongoClient(MONGODB_URI)
db = client["apple_retail_store"]


6. Insert data from each dataframes inside its corresponding collection

In [ ]:
db.products.drop()
db.categories.drop()
db.stores.drop()
db.warranties.drop()
db.sales.drop


<bound method Collection.drop of Collection(Database(MongoClient(host=['ac-zkn1wi5-shard-00-02.u9zhept.mongodb.net:27017', 'ac-zkn1wi5-shard-00-01.u9zhept.mongodb.net:27017', 'ac-zkn1wi5-shard-00-00.u9zhept.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, retrywrites=True, w='majority', appname='Cluster0', authsource='admin', replicaset='atlas-kvjhh5-shard-0', tls=True), 'apple_retail_store'), 'sales')>

In [ ]:
db.products.insert_many(products.to_dict("records"))
db.categories.insert_many(categories.to_dict("records"))
db.stores.insert_many(stores.to_dict("records"))
db.warranties.insert_many(warranties.to_dict("records"))


InsertManyResult([ObjectId('696146ca1446c7d285737848'), ObjectId('696146ca1446c7d285737849'), ObjectId('696146ca1446c7d28573784a'), ObjectId('696146ca1446c7d28573784b'), ObjectId('696146ca1446c7d28573784c'), ObjectId('696146ca1446c7d28573784d'), ObjectId('696146ca1446c7d28573784e'), ObjectId('696146ca1446c7d28573784f'), ObjectId('696146ca1446c7d285737850'), ObjectId('696146ca1446c7d285737851'), ObjectId('696146ca1446c7d285737852'), ObjectId('696146ca1446c7d285737853'), ObjectId('696146ca1446c7d285737854'), ObjectId('696146ca1446c7d285737855'), ObjectId('696146ca1446c7d285737856'), ObjectId('696146ca1446c7d285737857'), ObjectId('696146ca1446c7d285737858'), ObjectId('696146ca1446c7d285737859'), ObjectId('696146ca1446c7d28573785a'), ObjectId('696146ca1446c7d28573785b'), ObjectId('696146ca1446c7d28573785c'), ObjectId('696146ca1446c7d28573785d'), ObjectId('696146ca1446c7d28573785e'), ObjectId('696146ca1446c7d28573785f'), ObjectId('696146ca1446c7d285737860'), ObjectId('696146ca1446c7d2857378

## Part II - Insert Big Data

Okay let's do something harder now, the `sales.csv` contains more than 1 million rows. So it will harder to insert it in MongoDB since it will require much more compute power. Don't worry, there are some technics though 😉

<Note type="hint">

In `pd.read_csv` there is a parameter called `chunksize` this lets you process data in batch. Each chunk needs to be considered as an interator of a for loop:

```python

for chunk in df = pd.read_csv("path", chunksize=100_000):
    #CODE
```

Have fun with it!

</Note>

1. Normalize the columns of the CSV

In [ ]:
chunksize = 100_000

for chunk in pd.read_csv("/content/drive/MyDrive/JedhaBootcamp/sales.csv", chunksize=chunksize):
    chunk = normalize_columns(chunk)
    chunk["sale_id"] = chunk["sale_id"].astype(str)
    chunk["sale_date"] = chunk["sale_date"].astype(str)
    chunk["store_id"] = chunk["store_id"].astype(str)
    chunk["product_id"] = chunk["product_id"].astype(str)
    chunk["quantity"] = chunk["quantity"].astype(int)
    chunk["quantity"] = chunk["quantity"].astype(int)
    db.sales.insert_many(chunk.to_dict("records"))


2. Cast each columns to its right data type

3. Now insert that data inside your db

## Part 3 - Create indexes

1. For each collection, think and create indexes that might be relevant for analytics

In [ ]:
db.products.create_index(
    "product_id",
    unique=True,
    partialFilterExpression={
        "product_id": {
            "$exists": True,
            "$type": "string"
        }
    }
)



'product_id_1'

In [ ]:

db.products.create_index("category_id")


'category_id_1'

In [ ]:
db.products.count_documents({"product_id": None})


0

In [ ]:
db.categories.create_index("category_id", unique=True)


'category_id_1'

In [ ]:
db.stores.create_index("store_id", unique=True)
db.stores.create_index("country")


'country_1'

In [ ]:
db.sales.create_index([("sale_date", 1)])
db.sales.create_index([("product_id", 1)])
db.sales.create_index([("store_id", 1)])
db.sales.create_index([ ("store_id", 1), ("sale_date", 1)])


'sale_date_1_store_id_1'

In [ ]:
db.warranties.create_index("sale_id")
db.warranties.create_index("repair_status")


'repair_status_1'